# Submission SL-A: LightGBM with team features, Spikes state and move typicality

This notebook runs end to end from the raw competition files (`train.jsonl`, `test.jsonl`, `type_chart_gen2.csv`,
`sample_submission.csv`): preprocessing, move-damage table, species table, features, model and submission. It reads no
intermediate files (`data/processed/`, saved models or `best_params.json`); the tuned parameters are written in the notebook.

The project code comes from our repository ([mogliaaaa/FDS-Pokemon-Battles-Prediction-2026-Team-Plasma-Version](https://github.com/mogliaaaa/FDS-Pokemon-Battles-Prediction-2026-Team-Plasma-Version)), which is cloned
automatically when the notebook is not run from inside the repo (e.g. on Kaggle, where internet access must be on).

Sections: 1. setup, 2. data, 3. preprocessing, 4. features, 5. model, 6. validation (train only), 7. final fit and submission.

## 1. Setup

In [1]:
import subprocess
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

REPO_URL = "https://github.com/mogliaaaa/FDS-Pokemon-Battles-Prediction-2026-Team-Plasma-Version.git"
REPO_REF = "main"   # pin a commit hash or tag for the final, exactly reproducible run

def find_repo_root():
    # Local run: the notebook is in the repo (root or notebooks/)
    for p in (Path.cwd(), Path.cwd().parent):
        if (p / "src" / "features.py").exists():
            return p.resolve()
    # Kaggle (or anywhere else): clone the repository next to the notebook
    target = Path("PokePredict")
    if not (target / "src" / "features.py").exists():
        subprocess.run(["git", "clone", REPO_URL, str(target)], check=True)
        subprocess.run(["git", "-C", str(target), "checkout", REPO_REF], check=True)
    return target.resolve()

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))
commit = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"repo: {ROOT} | commit: {commit or 'unknown'}")

from sklearn.ensemble import VotingClassifier

from src import preprocessing as pp
from src.data import find_data_dir, load_jsonl, load_type_chart
from src.features import build_features, robust_columns
from src.models import ProbaBlend, get_model_specs, params_to_text, with_params
from src.move_damage import fit_move_damage
from src.move_typicality import with_move_typicality
from src.team_strength import with_team_strength
from src.validation import fold_accuracy, oof_proba

RUN_CV = True   # 5-fold CV on train before the final fit (set to False to only write the submission)

repo: .. | commit: 79f4359


## 2. Data

On Kaggle the files are in `/kaggle/input/...`, locally in `data/raw/`; `find_data_dir` checks both.

In [2]:
DATA_DIR = find_data_dir()
train = load_jsonl(DATA_DIR / "train.jsonl")
test = load_jsonl(DATA_DIR / "test.jsonl")
type_chart = load_type_chart(DATA_DIR / "type_chart_gen2.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
print(f"data dir: {DATA_DIR} | train: {len(train):,} | test: {len(test):,} | sample submission: {len(sample):,}")

data dir: ..\data\raw | train: 10,000 | test: 5,000 | sample submission: 5,000


## 3. Preprocessing

Same steps as `notebooks/02_preproc.ipynb` (code in `src/preprocessing.py` and `src/move_damage.py`):
- `team_frame`: P1's team and P2's lead, with normalised types and an under-level flag;
- `fit_move_damage`: expected damage of each move, estimated from the observed HP drops, so that moves with variable
  power (base power 0 in the data) get a realistic value. Fitted on train only;
- `timeline_frame`: one row per battle, side and turn, with the action labels and the expected damage;
- `pokemon_states`: last known state of every Pokémon seen in the 30 turns (KO, HP, vanished);
- `build_species_table`: base stats and types per species, computed on train only (species missing from train get the
  mean).

In [3]:
t0 = time.time()
damage_table = fit_move_damage(train)
species = pp.build_species_table(train)

def preprocess(battles):
    team = pp.team_frame(battles)
    timeline = pp.timeline_frame(battles, damage_table)
    states = pp.pokemon_states(timeline)
    return team, timeline, states

tables_train = preprocess(train)
tables_test = preprocess(test)
print(f"preprocessing done in {time.time() - t0:.0f}s")

preprocessing done in 7s


## 4. Features (v1 + team keys + Spikes state + move keys)

Same as `notebooks/03_features.ipynb` (`src/features.py`): 155 numeric features in 7 groups, plus the three Spikes columns.
There are also two pairs of text keys, the team keys and the move keys. They are not passed to the models directly: the
first steps of the pipeline encode them, fitted on the training data only.

In [4]:
options = dict(team_keys=True, hazards=True, move_keys=True)
X_train, groups = build_features(*tables_train, species, type_chart, **options)
X_test, _ = build_features(*tables_test, species, type_chart, **options)
y = X_train.pop("player_won")

KEYS, SPIKES, MOVE_KEYS = groups["team_keys"], groups["hazards"], groups["move_keys"]
BASE = [c for c in X_train.columns if c not in groups["species_cat"] + KEYS + SPIKES + MOVE_KEYS]
ROBUST = robust_columns(BASE)
assert len(BASE) == 155 and len(ROBUST) == 145

# The order of the columns is part of the model (LightGBM samples columns), so it is the same as in notebook 33
ALL_OLD, ROB_OLD = BASE + KEYS, ROBUST + KEYS
ALL_NEW, ROB_NEW = ALL_OLD + SPIKES + MOVE_KEYS, ROB_OLD + SPIKES + MOVE_KEYS
X_train, X_test = X_train[ALL_NEW], X_test[ALL_NEW]
assert list(X_train.columns) == list(X_test.columns)
print(f"train {X_train.shape} | test {X_test.shape}")
print("share of battles with Spikes on each side, train:", X_train[SPIKES[:2]].mean().round(3).to_dict(), "| test:", X_test[SPIKES[:2]].mean().round(3).to_dict())

train (10000, 162) | test (5000, 162)
share of battles with Spikes on each side, train: {'p1_spikes_on': 0.479, 'p2_spikes_on': 0.503} | test: {'p1_spikes_on': 0.466, 'p2_spikes_on': 0.502}


## 5. Model

Same tuned parameters of experiment 04 as the other submissions. Why we chose this model is explained in section 7.

In [5]:
# Tuned in notebooks/04_models.ipynb (experiment 04, RandomizedSearchCV, 5-fold CV, seed 42)
PARAMS = {
    'xgboost': {
        'model__colsample_bytree': 0.6856273981493477,
        'model__gamma': 0.010493314645539311,
        'model__learning_rate': 0.012145928554953287,
        'model__max_depth': 6,
        'model__min_child_weight': 1.3386261584543908,
        'model__n_estimators': 720,
        'model__reg_lambda': 3.5416296712651336,
        'model__subsample': 0.5993578407670862,
    },
    'lightgbm': {
        'model__colsample_bytree': 0.6286197472598067,
        'model__learning_rate': 0.006552992991573526,
        'model__min_child_samples': 106,
        'model__n_estimators': 1424,
        'model__num_leaves': 12,
        'model__reg_alpha': 0.6070874110546393,
        'model__reg_lambda': 2.7129089558470034,
        'model__subsample': 0.6629794526009424,
    },
    'catboost': {
        'model__bagging_temperature': 0.15071754396542947,
        'model__depth': 5,
        'model__iterations': 763,
        'model__l2_leaf_reg': 1.14876138417489,
        'model__learning_rate': 0.03898458749305194,
        'model__random_strength': 1.3551287236845648,
    },
    'gradient_boosting': {
        'model__learning_rate': 0.04760153910846178,
        'model__max_depth': 3,
        'model__max_features': None,
        'model__min_samples_leaf': 91,
        'model__n_estimators': 466,
        'model__subsample': 0.9314950036607718,
    },
}

LOGREG_PARAMS = {"model__C": 0.006690421166498805}

specs = get_model_specs()
for n in PARAMS:
    print(f"{n:18s} {params_to_text(PARAMS[n])}")

ENCODER_22 = {"train_unseen": 0.3}                                   # also submission 24
ENCODER_23 = {"counts": False, "train_unseen": 0.5}
R1_R3 = {"train_unseen": 0.3, "pairs": True, "per_side": True}       # submissions 27-29
LGBM = with_params(specs["lightgbm"], PARAMS["lightgbm"])
LOGREG = with_params(specs["logreg"], LOGREG_PARAMS)
VOTING = VotingClassifier([(n, with_params(specs[n], PARAMS[n])) for n in ("xgboost", "lightgbm", "catboost", "gradient_boosting")],
                          voting="soft", n_jobs=1)
LOGREG_WEIGHT = 0.25

# name, estimator, encoder, robust features
SPEC = [
    ("LightGBM", LGBM, R1_R3, False),
]
MEMBERS = [(name, with_move_typicality(with_team_strength(est, **enc)), ROB_NEW if robust else ALL_NEW) for name, est, enc, robust in SPEC]
BASELINE = [(name, with_team_strength(est, **enc), ROB_OLD if robust else ALL_OLD) for name, est, enc, robust in SPEC]
WEIGHTS = {SPEC[0][0]: 1.0}
assert abs(sum(WEIGHTS.values()) - 1) < 1e-12
model = ProbaBlend(MEMBERS, WEIGHTS)
pd.Series(WEIGHTS).round(4)

xgboost            colsample_bytree=0.6856, gamma=0.01049, learning_rate=0.01215, max_depth=6, min_child_weight=1.339, n_estimators=720, reg_lambda=3.542, subsample=0.5994
lightgbm           colsample_bytree=0.6286, learning_rate=0.006553, min_child_samples=106, n_estimators=1424, num_leaves=12, reg_alpha=0.6071, reg_lambda=2.713, subsample=0.663
catboost           bagging_temperature=0.1507, depth=5, iterations=763, l2_leaf_reg=1.149, learning_rate=0.03898, random_strength=1.355
gradient_boosting  learning_rate=0.0476, max_depth=3, max_features=None, min_samples_leaf=91, n_estimators=466, subsample=0.9315


LightGBM    1.0
dtype: float64

## 6. Validation (train only)

Stratified 5-fold CV with `shuffle=True`, on seeds 42, 7 and 123. Every step that learns from the data is part of the
model, so it is refitted on each training fold; the test set is never used.

**Check:** this notebook starts from the raw data, so it has to reproduce notebook 33 on the same folds: 0.6922 with the
new columns and 0.6888 for submission 27.

In [6]:
if RUN_CV:
    t0 = time.time()
    SEEDS = (42, 7, 123)
    def mean_oof(members, seeds):
        return np.concatenate([fold_accuracy(sum(WEIGHTS[name] * oof_proba(m, X_train[cols], y, seed=s) for name, m, cols in members), y, seed=s) for s in seeds])
    new, old = mean_oof(MEMBERS, SEEDS), mean_oof(BASELINE, SEEDS)
    gain = (new - old).mean() * 100
    print(f"with the new columns: {new.mean():.4f} (notebook 33: 0.6922) | submission 27: {old.mean():.4f} (notebook 33: 0.6888)")
    print(f"gain {gain:+.2f} points, better in {(new > old).sum()}/15 folds ({time.time() - t0:.0f}s)")
else:
    print("CV skipped (RUN_CV = False)")

with the new columns: 0.6922 (notebook 33: 0.6922) | submission 27: 0.6888 (notebook 33: 0.6888)
gain +0.34 points, better in 10/15 folds (47s)


## 7. Final fit and submission

**Approach.** The tuned LightGBM of experiment 04 with the R1 + R3 team encoder of submission 27, plus two new groups of
columns. This is exactly the configuration that passed the rule in notebook 33; section 6 repeats the measurement from
the raw data and has to reproduce notebook 33 (0.6922 against 0.6888).

**New columns (experiment 16, notebook 33).** Two groups of columns are added to the 155 features and the team features:

- Spikes state (`hazard_features`): the moves are replayed in order, so that each side knows whether it has Spikes on
  its field at turn 30. Spikes is laid on the opponent's side and Rapid Spin clears the user's side, unless the
  opponent's active Pokémon is a Ghost type. The 155 features count the Spikes moves but ignore Rapid Spin and the order.
  No target is used.
- Move typicality (`MoveTypicalityEncoder`, first step of the pipeline): how common the moves of each side are for its
  species, counted on the training battles of each fold and leaving each battle out of its own counts. No target is
  used, and test battles are never counted.

On LightGBM, paired on 15 folds against the model of submission 27: +0.34 points (10/15 folds) on seeds 42 / 7 / 123 and
+0.49 points (12/15) on the new seeds 8 / 9 / 10, with the rule fixed before running. With the new columns shuffled
across battles the result is -0.13 and -0.19 points. On their own the two groups give only +0.04 and +0.02 points: only
the combination passes, and we have no clear explanation for this, so part of the gain may be noise. All these folds
resample the same 10,000 battles.

The team encoder (`src/team_strength.py`) is described in experiments 11 and 13: it is refitted on each training fold,
and no battle ever sees its own label.

In [7]:
t0 = time.time()
model.fit(X_train, y)
pred = pd.Series(model.predict(X_test), index=X_test.index)
print(f"fitted and predicted in {time.time() - t0:.0f}s")

submission = sample[["battle_id"]].copy()
submission["player_won"] = submission["battle_id"].map(pred)
assert submission["player_won"].notna().all(), "some test battles have no prediction"
submission["player_won"] = submission["player_won"].astype(int)

kaggle_dir = Path("/kaggle/working")
out_path = kaggle_dir / "submission.csv" if kaggle_dir.exists() else ROOT / "submissions" / "submission_sl_lgbm.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(out_path, index=False)
print(f"saved: {out_path} | rows: {len(submission):,}")
print("predicted P1 win rate:", round(submission["player_won"].mean(), 4))

# Descriptive only: how much the predictions change against submission 27, if its file is here
previous = ROOT / "submissions" / "submission_ts2_lgbm.csv"
if previous.exists():
    other = pd.read_csv(previous).set_index("battle_id")["player_won"].reindex(submission["battle_id"]).to_numpy()
    print(f"differs from submission 27 on {(other != submission['player_won'].to_numpy()).mean():.1%} of the battles")
submission.head()

fitted and predicted in 7s
saved: ..\submissions\submission_sl_lgbm.csv | rows: 5,000
predicted P1 win rate: 0.5202
differs from submission 27 on 5.8% of the battles


,battle_id,player_won
0,0,1
1,1,1
2,2,0
3,3,0
4,4,1
